<a href="https://colab.research.google.com/github/ranygo12-pixel/AWS/blob/main/Bedrock/Bedrock04/Bedrock04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**검색된 문서**

In [ ]:
pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.2 MB/s eta 0:00:00


In [ ]:
import os
import boto3
from google.colab import userdata

# 1. 환경 변수 설정 및 클라이언트 생성
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'ap-northeast-2'

kb_id = userdata.get('KB_ID')
bedrock = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')

# 2. 함수 정의 (여기까지만 들여쓰기가 적용됩니다)
def check_code_style(code, kb_id):
    """Knowledge Base를 사용해 코드 스타일 검사"""
    prompt = f"""
    다음 코드가 PEP8 스타일 가이드를 위반하는 부분이 있다면 알려주세요.
    위반 사항이 없으면 "통과"라고 답변해주세요.

    코드:
    {code}
    """

    response = bedrock.retrieve_and_generate(
        input={'text': prompt},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': 'global.anthropic.claude-sonnet-4-6'
            }
        }
    )
    return response['output']['text']


# 3. 실제 실행 테스트 (★들여쓰기를 맨 앞으로 붙여서 함수 바깥으로 뺐습니다★)
print("--- 1. 단순 연결 및 응답 테스트 시작 ---")
try:
    response_raw = bedrock.retrieve_and_generate(
        input={'text': "안녕하세요, 연결 테스트입니다."},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': 'global.anthropic.claude-sonnet-4-6'
            }
        }
    )
    print("✅ AWS 연결 응답 성공! 결과:")
    print(response_raw['output']['text'])
except Exception as e:
    print(f"❌ 연결 실패 에러: {e}")

print("\n--- 2. 코드 스타일 검사 테스트 시작 ---")
test_code = """
def add(a,b):
    return a+b
"""

try:
    result = check_code_style(test_code, kb_id)
    print("✅ 코드 검사 결과:")
    print(result)
except Exception as e:
    print(f"❌ 코드 검사 실패 에러: {e}")

--- 1. 단순 연결 및 응답 테스트 시작 ---
✅ AWS 연결 응답 성공! 결과:
안녕하세요! 연결이 정상적으로 되었습니다. 무엇이든 궁금한 점이 있으시면 질문해 주세요. 😊

--- 2. 코드 스타일 검사 테스트 시작 ---
✅ 코드 검사 결과:
해당 코드에서 PEP 8 위반 사항이 있습니다:

1. **함수 인자 사이 공백 누락**: `add(a,b)`에서 쉼표 뒤에 공백이 없습니다. → `add(a, b)`로 수정해야 합니다.
2. **연산자 주변 공백 누락**: `a+b`에서 `+` 연산자 앞뒤에 공백이 없습니다. → `a + b`로 수정해야 합니다.

올바른 코드:
```python
def add(a, b):
    return a + b
```


In [ ]:
def check_with_sources(code, kb_id):
    """코드 검사 + 어떤 문서가 사용됐는지 출력"""

    # Explicitly create a bedrock-agent-runtime client for retrieve_and_generate
    bedrock_agent_client = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')

    response = bedrock_agent_client.retrieve_and_generate(
        input={'text': f"다음 코드의 PEP8 위반사항을 찾아줘:\n{code}"},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': 'global.anthropic.claude-sonnet-4-6'
            }
        }
    )

    # 사용된 문서 출력
    print("📚 참고한 문서:")
    for result in response.get('retrievedResults', []):
        score = result['score']
        content = result['content']['text'][:150]
        print(f"   - 관련성 {score:.2f}: {content}...")

    print("\n🤖 최종 답변:")
    return response['output']['text']

# 실행
code = "def calculate(x,y): return x+y"
print(check_with_sources(code, kb_id))

📚 참고한 문서:

🤖 최종 답변:
해당 코드의 PEP 8 위반 사항은 다음과 같습니다:

1. **함수 인자 사이 공백 없음**
   - 위반: `calculate(x,y)`
   - 수정: `calculate(x, y)` — 쉼표 뒤에 공백 추가

2. **연산자 주변 공백 없음**
   - 위반: `x+y`
   - 수정: `x + y` — 연산자 양쪽에 공백 추가

3. **함수 본문이 같은 줄에 작성됨**
   - 위반: `def calculate(x,y): return x+y` (한 줄에 작성)
   - 수정: `return` 문은 다음 줄에 4 spaces 들여쓰기로 작성해야 함

**수정된 코드:**
```python
def calculate(x, y):
    return x + y
```


In [ ]:
import boto3

def check_with_sources(code, kb_id):
    """코드 검사 + 어떤 문서가 사용됐는지 출력 (Citations 버전)"""

    # Explicitly create a bedrock-agent-runtime client for retrieve_and_generate
    bedrock_agent_client = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')

    response = bedrock_agent_client.retrieve_and_generate(
        input={'text': f"다음 코드의 PEP8 위반사항을 찾아줘:\n{code}"},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,  # 1. 오타 수정 (kd_id -> kb_id)
                'modelArn': 'global.anthropic.claude-sonnet-4-6'
            }
        }
    )

    # 사용된 문서 출력
    print("📚 참고한 문서 (Citations):")
    for citation in response.get('citations', []):
        for ref in citation.get('retrievedReferences', []):
            content = ref['content']['text'][:150]
            print(f"   - {content.strip()}...")

    print("\n🤖 최종 답변:")
    return response['output']['text']


# ========================================================
# 2. 실행 테스트 영역 (들여쓰기 맨 앞으로 복구 및 호출)
# ========================================================

# 테스트할 위반 코드
code = "def calculate(x,y): return x+y"

# 함수 호출 및 출력
try:
    final_output = check_with_sources(code, kb_id)
    print(final_output)
except Exception as e:
    print(f"❌ 실행 중 에러 발생: {e}")

📚 참고한 문서 (Citations):
   - PEP 8 - Python Code Style Guide  ## Naming Conventions - 변수명: snake_case 사용 (예: user_name, total_count) - 함수명: snake_case 사용 (예: calculate_total, get_...
   - PEP 8 - Python Code Style Guide  ## Naming Conventions - 변수명: snake_case 사용 (예: user_name, total_count) - 함수명: snake_case 사용 (예: calculate_total, get_...

🤖 최종 답변:
해당 코드의 PEP 8 위반사항은 다음과 같습니다:

1. **함수명**: `calculate`는 괜찮지만, 인자 사이 공백 없음
   - `calculate(x,y)` → `calculate(x, y)` (쉼표 뒤 공백 필요)

2. **들여쓰기 및 줄 분리**: 함수 본문이 같은 줄에 작성됨
   - `return x+y`는 다음 줄에 4 spaces 들여쓰기로 작성해야 함

3. **연산자 주변 공백 없음**:
   - `x+y` → `x + y` (연산자 양쪽에 공백 필요)

수정된 코드:
```python
def calculate(x, y):
    return x + y
```


코드 스타일 검사 함수

In [ ]:
import boto3

def style_check(code, kb_id):
    """코드 스타일 검사 후 위반 사항 리스트 반환"""

    # Ensure bedrock-agent-runtime client is used
    bedrock_agent_client_for_rag = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')

    prompt = f"""
    당신은 코드 스타일 검사기입니다. 다음 코드에서 PEP8 또는 일반적인 스타일 규칙을 위반한 부분을 찾아주세요.

    형식:
    - [라인번호] 위반 유형: 설명

    코드:
    {code}
    """

    response = bedrock_agent_client_for_rag.retrieve_and_generate(
        input={'text': prompt},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': 'global.anthropic.claude-sonnet-4-6'
            }
        }
    )

    # 1. 여기서 return을 하면서 함수가 깔끔하게 끝납니다.
    return response['output']['text']


# ========================================================
# 2. 실행 테스트 영역 (들여쓰기를 모두 맨 앞으로 당겼습니다!)
# ========================================================

# 단일 코드 테스트
bad_code = """
def Sum(a,b):
    return a+b
"""
print("=== 단일 bad_code 테스트 결과 ===")
print(style_check(bad_code, kb_id))
print("-" * 40)


# 샘플 리스트 다중 테스트 수행
samples = [
    """def add(a,b): return a+b""",   # 한 줄에 여러 구문
    """def GetUserData(): pass""",    # 함수명 대문자 시작
    """import os,sys""",              # 여러 import 한 줄에
    """x = 1\ny = 2\nz=x+y""",        # 공백 부족
    """def long_function_name_very_long_parameter_list(param1, param2, param3, param4, param5): pass"""
]

print("=== 여러 샘플 코드 순회 검사 시작 ===")
for i, sample_code in enumerate(samples, 1):
    print(f"\n[샘플 {i}] 검사할 코드: {sample_code}")

    try:
        result = style_check(sample_code, kb_id)
        print(f"결과:\n{result}")
    except Exception as e:
        print(f"❌ 샘플 {i} 검사 중 에러 발생: {e}")
    print("=" * 30)

=== 단일 bad_code 테스트 결과 ===
다음과 같은 PEP 8 스타일 위반 사항이 발견되었습니다:

- [1] **함수명 네이밍 규칙 위반**: 함수명 `Sum`은 PascalCase로 작성되어 있으나, 함수명은 snake_case를 사용해야 합니다. → `sum_values` 또는 `add` 등으로 변경 권장
- [1] **공백 누락**: 함수 파라미터 `a,b` 사이의 쉼표 뒤에 공백이 없습니다. → `a, b`로 변경 필요
- [2] **연산자 주변 공백 누락**: `a+b` 표현식에서 `+` 연산자 양쪽에 공백이 없습니다. → `a + b`로 변경 필요
----------------------------------------
=== 여러 샘플 코드 순회 검사 시작 ===

[샘플 1] 검사할 코드: def add(a,b): return a+b
결과:
다음과 같은 PEP 8 스타일 위반 사항이 발견되었습니다:

- [1] **공백 누락**: 함수 인자 사이의 쉼표(,) 뒤에 공백이 없습니다. `a,b` → `a, b`
- [1] **공백 누락**: 연산자 `+` 양쪽에 공백이 없습니다. `a+b` → `a + b`
- [1] **단일 라인 함수 본문**: 함수 정의와 본문(`return`)이 같은 줄에 작성되었습니다. 함수 본문은 다음 줄에 들여쓰기(4 spaces)하여 작성해야 합니다.

올바른 코드 예시:
```python
def add(a, b):
    return a + b
```

[샘플 2] 검사할 코드: def GetUserData(): pass
결과:
코드 스타일 위반 사항:

- [1] 함수명 명명 규칙 위반: 함수명은 PascalCase(GetUserData)가 아닌 snake_case를 사용해야 합니다. 올바른 함수명은 `get_user_data`입니다.

[샘플 3] 검사할 코드: import os,sys
결과:
- [1] 임포트 분리 위반: 여러 모듈을 한 줄에 같이 임포트하면 안 됩니다. `os`와 `sys`는 각각 별

In [ ]:
import boto3

def check_security(code, kb_id):
    """보안 취약점 검사"""

    # Explicitly create a bedrock-agent-runtime client for retrieve_and_generate
    bedrock_agent_client = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')

    response = bedrock_agent_client.retrieve_and_generate(
        input={'text': f"""
                        다음 코드에서 보안 취약점을 찾아주세요. 특히 SQL Injection, XSS, 하드코딩된 비밀번호를 중점적으로 검사해주세요.

                        코드:
                        {code}
                        취약점이 있으면 위치, 유형, 심각도, 수정 제안을 포함해 주세요.
                        """},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': 'global.anthropic.claude-sonnet-4-6'
            }
        }
    )

    return response['output']['text']

# ========================================================
# 수정 구간: 들여쓰기를 맨 앞으로 맞추고 끊긴 문자열을 닫아줬습니다.
# ========================================================

# 취약한 코드 테스트
vulnerable_code = """
def get_user_by_name(username):
    query = "SELECT * FROM users WHERE name = '" + username + "'"
"""  # 👈 닫는 따옴표 추가

# 결과 출력 추가
print(check_security(vulnerable_code, kb_id))

## 보안 취약점 분석 결과

### 🚨 취약점 발견: SQL Injection

**위치:** `get_user_by_name` 함수 내 `query` 변수 생성 부분

**유형:** SQL Injection (A03:2021 – Injection)

**심각도:** 🔴 높음 (Critical)

**문제점:**
사용자 입력값(`username`)을 검증 없이 SQL 쿼리 문자열에 직접 연결(concatenation)하고 있습니다.
예를 들어, 공격자가 `username`에 `' OR '1'='1` 같은 값을 입력하면 모든 사용자 정보가 노출될 수 있습니다.

**공격 예시:**
```python
# 입력값: ' OR '1'='1
# 생성된 쿼리: SELECT * FROM users WHERE name = '' OR '1'='1'
# → 모든 사용자 정보 반환
```

---

### ✅ 수정 제안: Parameterized Query (파라미터화된 쿼리) 사용

```python
def get_user_by_name(username):
    query = "SELECT * FROM users WHERE name = %s"
    cursor.execute(query, (username,))  # 파라미터 바인딩
```

- 사용자 입력값을 쿼리 문자열과 분리하여 SQL Injection 방지
- DB 라이브러리(예: `psycopg2`, `sqlite3`, `pymysql`)에서 제공하는 파라미터 바인딩 방식을 항상 사용해야 합니다

---

### 기타 검토 사항
| 항목 | 결과 |
|------|------|
| XSS | 해당 코드에서는 발견되지 않음 (HTML 출력 없음) |
| 하드코딩된 비밀번호 | 해당 코드에서는 발견되지 않음 |
| SQL Injection | ⚠️ 발견됨 → 즉시 수정 필요 |


**취약점 리포트 생성**

In [ ]:
import boto3

def generate_security_report(code, kb_id):
    """보안 취약점 리포트 (JSON 형식) 생성"""

    # Explicitly create a bedrock-agent-runtime client for retrieve_and_generate
    bedrock_agent_client = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')

    response = bedrock_agent_client.retrieve_and_generate(
        input={'text': f"""
        다음 코드의 보안 취약점을 분석하고 JSON 형식으로 보고서를 작성해주세요.

        형식:
        {{
            "vulnerabilities": [
                {{
                    "line": 라인번호,
                    "type": "취약점 유형",
                    "severity": "CRITICAL/HIGH/MEDIUM/LOW",
                    "description": "설명",
                    "suggestion": "수정 제안"
                }}
            ],
            "summary": "전체 평가"
        }}

        코드:
        {code}
        """},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': 'global.anthropic.claude-sonnet-4-6'
            }
        }
    )

    return response['output']['text']


# ========================================================
# 수정 및 추가 구간: 들여쓰기를 맞추고, vulnerable_code를 정의했습니다.
# ========================================================

# 테스트용 취약한 샘플 코드 정의 (앞 단계에서 썼던 코드 활용)
vulnerable_code = """
def get_user_by_name(username):
    query = "SELECT * FROM users WHERE name = '" + username + "'"
    return query
"""

# 결과를 파일로 저장
report = generate_security_report(vulnerable_code, kb_id)
with open('security_report.json', 'w', encoding='utf-8') as f:
    f.write(report)

print("✅ 리포트 저장 완료: security_report.json")

✅ 리포트 저장 완료: security_report.json


In [ ]:
import boto3
from pathlib import Path
import os
from google.colab import userdata

# Colab Secrets에서 AWS 키를 안전하게 가져와 환경 변수에 등록합니다.`
try:
    os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('AWS_ACCESS_KEY_ID')
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('AWS_SECRET_ACCESS_KEY')
    os.environ["AWS_DEFAULT_REGION"] = "ap-northeast-2"
    KB_ID = userdata.get('KB_ID')
    print("✅ Colab Secrets로부터 AWS 자격 증명 및 KB_ID 로드 완료!")
except Exception as e:
    print("❌ 에러: Colab Secrets 설정을 확인하세요 (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, KB_ID)")


def style_check(code, kb_id):
    """지식 기반을 활용한 PEP8 스타일 검사"""
    client = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')
    prompt = f"""
당신은 코드 스타일 검사기입니다. 다음 코드에서 PEP8 또는 일반적인 스타일 규칙을 위반한 부분을 찾아주세요.
형식은 반드시 아래 형식을 지켜주세요:
[라인번호] 위반 유형: 설명

코드:
{code}
"""
    response = client.retrieve_and_generate(
        input={'text': prompt},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': 'global.anthropic.claude-sonnet-4-6'
            }
        }
    )
    return response['output']['text']


def check_security(code, kb_id):
    client = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')

    # 1단계: KB에서 관련 문서 검색
    retrieve_response = client.retrieve(
        knowledgeBaseId=kb_id,
        retrievalQuery={'text': 'OWASP security vulnerabilities SQL Injection hardcoded credentials'}
    )

    # 검색된 문서 내용 추출
    context = "\n".join([r['content']['text'] for r in retrieve_response['retrievalResults']])

    # 2단계: 모델에 직접 질문
    bedrock = boto3.client('bedrock-runtime', region_name='ap-northeast-2')
    prompt = f"""아래 보안 가이드를 참고하여 코드의 보안 취약점을 분석해주세요.

보안 가이드:
{context}

분석할 코드:
{code}

각 항목을 위치 / 유형 / 심각도 / 개선 방법 형식으로 작성해주세요.
"""
    import json
    response = bedrock.invoke_model(
        modelId='anthropic.claude-3-5-sonnet-20241022-v2:0',
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 2000,
            "messages": [{"role": "user", "content": prompt}]
        })
    )
    result = json.loads(response['body'].read())
    return result['content'][0]['text']


def full_project_review(project_path, kb_id):
    results = {}
    total_lines = 0

    for py_file in Path(project_path).glob('**/*.py'):
        if '.venv' in py_file.parts or '__pycache__' in py_file.parts:
            continue

        try:
            with open(py_file, 'r', encoding='utf-8') as f:
                code = f.read()
        except Exception:
            with open(py_file, 'r', encoding='cp949', errors='ignore') as f:
                code = f.read()

        if len(code.splitlines()) < 50:
            continue

        print(f"🔄 현재 검사 진행 중: {py_file.name}")

        style_result = style_check(code, kb_id)
        security_result = check_security(code, kb_id)

        file_lines = len(code.splitlines())
        total_lines += file_lines

        results[py_file.name] = {
            'style': style_result,
            'security': security_result,
            'lines': file_lines
        }

    print(f"\n📊 총 검사된 코드 라인 수: {total_lines}줄")

    report_output_path = f"{project_path}/project_review_report.md"

    report_content = f"""# 🚀 CodeBuddy 통합 코드 리뷰 리포트

- **총 검사 라인 수:** {total_lines}줄
- **검사 도구:** Amazon Bedrock (Claude 4.5 Sonnet v2)
- **기반 문서:** PEP8 가이드라인 & OWASP Top 10

---

"""
    for filename, data in results.items():
        report_content += f"""## 📄 파일: {filename} ({data['lines']}줄)

### 🔹 1. 스타일 검사 결과 (PEP8)
{data['style']}

### 🔹 2. 보안 취약점 검사 결과
{data['security']}

---

"""

    with open(report_output_path, 'w', encoding='utf-8') as f:
        f.write(report_content)

    print(f"✅ 종합 리포트가 구글 드라이브에 성공적으로 저장되었습니다:\n👉 {report_output_path}")
    return results


# ==========================================
# [4단계] 실행 제어 부문
# ==========================================

PROJECT_PATH = "/content/drive/MyDrive/my_project_folder"

if KB_ID:
    full_project_review(PROJECT_PATH, KB_ID)
else:
    print("❌ 에러: Secrets에서 'KB_ID'를 성공적으로 가져오지 못했습니다.")

### 시맨틱 검사

In [ ]:
def semantic_search_query(kb_id, question):
    """시맨틱 검색 사용"""

    # Explicitly create a bedrock-agent-runtime client for retrieve_and_generate
    bedrock_agent_client = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')

    response = bedrock_agent_client.retrieve_and_generate(
        input={'text': question},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': 'global.anthropic.claude-sonnet-4-6',
                'retrievalConfiguration': {
                    'vectorSearchConfiguration': {
                        'numberOfResults': 5,
                        'overrideSearchType': 'SEMANTIC'  #  ← SEMANTIC
                    }
                }
            }
        }
    )

    return response['output']['text']

# 비교 실험
question = "사용자 입력을 SQL 쿼리에 직접 넣으면 위험해?"
print("시맨틱 결과:", semantic_search_query(kb_id, question))

시맨틱 결과: 네, 매우 위험합니다. 사용자 입력을 SQL 쿼리에 직접 삽입하면 **SQL Injection** 공격에 노출될 수 있습니다. 이는 신뢰할 수 없는 데이터가 쿼리나 명령의 일부로 해석되어 실행되는 문제로, OWASP Top 10에서 **A03:2021 – Injection**으로 분류된 주요 보안 취약점입니다.

공격자는 이를 통해 데이터베이스를 무단으로 조회, 수정, 삭제하거나 시스템을 장악할 수 있습니다. 따라서 Prepared Statement나 Parameterized Query 같은 안전한 방법을 사용하는 것이 권장됩니다.


### 검색 결과 수 TopK 튜닝

In [ ]:
import os
import boto3
from google.colab import userdata

# 1. 환경 변수 설정 및 클라이언트 생성 (Ensure kb_id is defined)
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'ap-northeast-2'

kb_id = userdata.get('KB_ID')

def test_topk(kb_id, question, topk):
    bedrock_agent_client = boto3.client('bedrock-agent-runtime',\
                             region_name='ap-northeast-2')

    response = bedrock_agent_client.retrieve_and_generate(
        input={'text': question},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': 'global.anthropic.claude-sonnet-4-6',
                'retrievalConfiguration': {
                    'vectorSearchConfiguration': {
                        'numberOfResults': topk  # ← 변경!
                    }
                }
            }
        }
    )
    return response['output']['text']

for k in [3, 5, 10]:
    result = test_topk(kb_id, "파이썬 변수명 규칙", k)
    print(f"TopK={k} 결과 길이: {len(result)}자")

TopK=3 결과 길이: 284자
TopK=5 결과 길이: 268자
TopK=10 결과 길이: 295자


### Relevance Score 필터링

In [ ]:
import boto3

# Bedrock 런타임 클라이언트 생성 (서울 리전)
bedrock_runtime_client = boto3.client('bedrock-runtime', region_name='ap-northeast-2')

def get_claude_response(user_message):
    response = bedrock_runtime_client.converse(
        modelId='global.anthropic.claude-sonnet-4-6',
        messages=[{"role": "user", "content": [{"text": user_message}]}]
    )
    return response['output']['message']['content'][0]['text']

def filter_by_relevance(kb_id, question, threshold=0.7):
    """관련성 점수가 threshold 미만인 문서는 제외"""

    # Bedrock Agent Runtime 클라이언트를 명시적으로 생성
    bedrock_agent_client = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')

    # 먼저 검색 결과 가져오기 (생성 없이)
    response = bedrock_agent_client.retrieve(
        knowledgeBaseId=kb_id,
        retrievalQuery={'text': question},
        retrievalConfiguration={
            'vectorSearchConfiguration': {
                'numberOfResults': 10
            }
        }
    )

    # 점수 높은 문서만 필터링
    filtered_docs = []
    for r in response['retrievalResults']:
        if r['score'] >= threshold:
            filtered_docs.append(r['content']['text'])

    # 필터링된 문서로 LLM 호출
    context = "\n\n".join(filtered_docs)
    final_prompt = f"참고 문서:\n{context}\n\n질문: {question}"

    return get_claude_response(final_prompt)  # 2장 함수 재사용

# 테스트
print(filter_by_relevance(kb_id, "SQL Injection 방지", threshold=0.7))


# SQL Injection 방지 가이드

## SQL Injection이란?

SQL Injection은 악의적인 SQL 코드를 입력값에 삽입하여 데이터베이스를 조작하는 공격 기법입니다.

---

## 🔴 취약한 코드 예시

```python
# 위험한 코드 - 절대 사용 금지
query = "SELECT * FROM users WHERE username = '" + username + "' AND password = '" + password + "'"
cursor.execute(query)
```

```sql
-- 공격 예시
username: admin' --
password: anything

-- 실행되는 쿼리
SELECT * FROM users WHERE username = 'admin' --' AND password = 'anything'
```

---

## ✅ 방지 방법

### 1. **Prepared Statement (파라미터화된 쿼리)** - 가장 중요

```python
# Python (mysql-connector)
query = "SELECT * FROM users WHERE username = %s AND password = %s"
cursor.execute(query, (username, password))

# Python (sqlite3)
query = "SELECT * FROM users WHERE id = ?"
cursor.execute(query, (user_id,))
```

```java
// Java - PreparedStatement
String query = "SELECT * FROM users WHERE username = ? AND password = ?";
PreparedStatement pstmt = connection.prepareStatement(query);
pstmt.setString(1, username);
pstmt.setString(2, password);
ResultSet rs = ps

### 여러 언어 지원

In [ ]:
%%bash

# 1. JavaScript Airbnb Style Guide 파일 생성
cat << 'EOF' > javascript_airbnb.txt
Airbnb JavaScript Style Guide

## Variables
- `const` for all references; avoid `var`.
- `let` if you plan on reassigning your variable.

## Objects
- Use trailing commas.

## Arrays
- Use array literals.

## Functions
- Use named function expressions.
EOF

# 2. Java Google Style Guide 파일 생성
cat << 'EOF' > java_google.txt
Google Java Style Guide

## Naming
- Class names are written in UpperCamelCase.
- Method names are written in lowerCamelCase.

## Formatting
- Indentation is 2 spaces.
- Column limit is 100 characters.

## Semicolons
- Semicolons are usually inserted automatically by the lexer.
EOF

In [ ]:
def check_multilang(code, language):
    """언어별 스타일 가이드 검사"""
    prompt = f"""
            {language} 코드 스타일 가이드에 따라 다음 코드를 검사해주세요.
            언어: {language}
            코드:
            {code}
            """
    return ask_knowledge_base(kb_id, prompt)

In [ ]:
import boto3

def ask_knowledge_base(kb_id, prompt):
    """Knowledge Base에 질문하여 답변을 반환하는 헬퍼 함수"""
    bedrock_agent_client = boto3.client('bedrock-agent-runtime', region_name='ap-northeast-2')

    response = bedrock_agent_client.retrieve_and_generate(
        input={'text': prompt},
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': 'global.anthropic.claude-sonnet-4-6'
            }
        }
    )
    return response['output']['text']

In [ ]:
# JavaScript 코드 검사
js_code = """
function helloWorld() {
  console.log('Hello, World!');
}
"""
print("--- JavaScript 코드 검사 ---")
print(check_multilang(js_code, "JavaScript"))

# Java 코드 검사
java_code = """
public class HelloWorld {
  public static void main(String[] args) {
    System.out.println("Hello, World!");
  }
}
"""

--- JavaScript 코드 검사 ---
Airbnb JavaScript Style Guide를 기준으로 제공된 코드를 검사한 결과는 다음과 같습니다:

**✅ 문제없는 부분:**
- 함수 선언 자체는 이름이 있는 함수(named function)로 작성되어 있습니다.

**⚠️ 개선이 필요한 부분:**

1. **함수 선언 방식**: Airbnb 스타일 가이드에서는 named function **expression** 사용을 권장합니다.
   ```javascript
   // ❌ 현재 코드
   function helloWorld() {
     console.log('Hello, World!');
   }

   // ✅ 권장 방식
   const helloWorld = function helloWorld() {
     console.log('Hello, World!');
   };
   ```

2. **변수 선언**: 함수를 변수에 할당할 경우, `var` 대신 `const`를 사용해야 합니다. 이 함수는 재할당될 필요가 없으므로 `const`가 적합합니다.

**총평:** 코드 자체의 로직은 문제없으나, 함수 표현식(function expression) 방식과 `const` 키워드를 활용하는 방식으로 리팩토링하는 것이 Airbnb JavaScript Style Guide에 더 부합합니다.


### Redis

In [ ]:
# 로컬 Redis (또는 AWS ElastiCache)
!pip install redis
!apt-get update
!apt-get install redis-server
# Start Redis server in the background
!redis-server --daemonize yes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 8.2 MB/s eta 0:00:00
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,070 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,292 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:1

In [ ]:
import redis
import hashlib
import json
import time # Add time import

# Redis 연결
cache = redis.Redis(host='localhost', port=6379, decode_responses=True)

def cached_code_review(code, kb_id, ttl_seconds=3600):
    """캐시된 코드 검사 (1시간 TTL)"""

    # 코드 해시 생성
    code_hash = hashlib.md5(code.encode()).hexdigest()
    cache_key = f"code_review:{code_hash}"

    # 캐시 확인
    cached = cache.get(cache_key)
    if cached:
        print("✅ 캐시에서 결과 반환 (비용 0원)")
        return cached

    # 캐시 미스 → 실제 검사
    print("🔍 RAG 검사 실행...")
    result = style_check(code, kb_id)  # 4장 함수

    # 캐시 저장
    cache.setex(cache_key, ttl_seconds, result)

    return result

In [ ]:
# 테스트
code = "def add(a, b): return a + b"

# Add a delay to ensure Redis server is ready
time.sleep(5)

first = cached_code_review(code, kb_id)   		# RAG 실행
second = cached_code_review(code, kb_id)  		# 캐시에서 반환

ConnectionError: Error 111 connecting to localhost:6379. Connection refused.

### 성능 측정 함수

In [ ]:
import time

def measure_performance(kb_id, code):
    """검사 시간 측정"""

    start = time.time()
    result = style_check(code, kb_id)
    elapsed = time.time() - start

    print(f"⏱️ 소요 시간: {elapsed:.2f}초")
    print(f"📄 결과 길이: {len(result)}자")

    # 토큰 수 예측 (선택)
    tokens = len(code.split()) + len(result.split())
    print(f"💰 예상 비용: 약 ${tokens * 0.00001:.4f}")

In [ ]:
# 다양한 크기 코드로 테스트
sizes = [100, 500, 1000, 5000]
for size in sizes:
    code = "def test():\n    pass\n" * (size // 20)
    print(f"\n코드 크기: {size}줄")
    measure_performance(kb_id, code)